# Tarrant CAD — PropertyData_2025 ⨝ PropertyDataSupplemental_{R,C,M,P}_2025

Joins the bronze outputs of the two upstream notebooks:
- `PropertyData_2025.csv` (from `tarrant_property_data_etl.ipynb`) — one row per account, all property types (`RP` = R / C / M / P), ~2.0M rows
- `PropertyDataSupplemental_{R,C,M,P}_2025.csv` (from `tarrant_property_data_supplemental_etl.ipynb`) — one file per property type, stacked into a single supplemental table

**Join key:** `RP` + `Appraisal_Year` + `Account_Num` ↔ `RP` + `AppraisalYear` + `AccountNumber`.
`RP` is part of the key because a couple of accounts appear in two supplemental files (e.g. `01073451` in both R and C); matching on `RP` picks the row for the account's actual type in PropertyData.

**Join type:** left — every PropertyData row is kept; supplemental columns are null where there's no match.
`SiteNumber` only exists in the C file, so it's null for R / M / P rows.

## 1. Setup

In [3]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

OUTPUT_DIR = Path("../output")

PROPERTY_FILE = OUTPUT_DIR / "PropertyData_2025.csv"
SUPPLEMENTAL_FILES = {rp: OUTPUT_DIR / f"PropertyDataSupplemental_{rp}_2025.csv" for rp in ("R", "C", "M", "P")}
OUTPUT_FILE = OUTPUT_DIR / "PropertyData_Joined_2025.csv"

for f in (PROPERTY_FILE, *SUPPLEMENTAL_FILES.values()):
    print(f.resolve(), f"{f.stat().st_size / 1e6:,.1f} MB")

/Users/user2/PTP/CAD Website/Tarrant/output/PropertyData_2025.csv 782.4 MB
/Users/user2/PTP/CAD Website/Tarrant/output/PropertyDataSupplemental_R_2025.csv 352.4 MB
/Users/user2/PTP/CAD Website/Tarrant/output/PropertyDataSupplemental_C_2025.csv 32.1 MB
/Users/user2/PTP/CAD Website/Tarrant/output/PropertyDataSupplemental_M_2025.csv 166.5 MB
/Users/user2/PTP/CAD Website/Tarrant/output/PropertyDataSupplemental_P_2025.csv 13.1 MB


## 2. Load
Read everything as string so leading zeros in account numbers survive. Strip padding on the key columns so both sides compare cleanly.

In [4]:
prop = pd.read_csv(PROPERTY_FILE, dtype=str, keep_default_na=False, encoding="latin-1")
for col in ("RP", "Appraisal_Year", "Account_Num"):
    prop[col] = prop[col].str.strip()

print("PropertyData:", prop.shape)
prop["RP"].value_counts()

PropertyData: (1996478, 56)


RP
M    1207764
R     659361
P      68227
C      61126
Name: count, dtype: int64

In [5]:
supp_parts = []
for rp, path in SUPPLEMENTAL_FILES.items():
    part = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="latin-1")
    part.columns = part.columns.str.strip()
    for col in ("RP", "AppraisalYear", "AccountNumber"):
        part[col] = part[col].str.strip()
    assert (part["RP"] == rp).all(), f"{path.name} contains RP values other than {rp}"
    print(f"{rp}: {part.shape}")
    supp_parts.append(part)

supp = pd.concat(supp_parts, ignore_index=True)
print("Supplemental (stacked):", supp.shape)

R: (658409, 28)
C: (61111, 29)
M: (1207563, 28)
P: (68227, 28)
Supplemental (stacked): (1995310, 29)


## 3. Key checks

In [6]:
SUPP_KEY = ["RP", "AppraisalYear", "AccountNumber"]
PROP_KEY = ["RP", "Appraisal_Year", "Account_Num"]

assert not prop.duplicated(PROP_KEY).any(), "duplicate keys in PropertyData"
assert not supp.duplicated(SUPP_KEY).any(), "duplicate keys in Supplemental"

cross_type = supp[supp.duplicated(["AppraisalYear", "AccountNumber"], keep=False)]
print(f"Accounts in more than one supplemental file: {cross_type['AccountNumber'].nunique():,}")
cross_type[SUPP_KEY + ["SiteClassDescr", "LandUseCode"]]

Accounts in more than one supplemental file: 2


,RP,AppraisalYear,AccountNumber,SiteClassDescr,LandUseCode
73201,R,2025,01073451,Residential - Vacant Land ...,Residential 3H ...
618567,R,2025,42722444,Residential - Agricultural ...,Residential 4B1 ...
664236,C,2025,01073451,Warehouse-Storage ...,Warehouse-Non Business Park ...
714289,C,2025,42722444,Exempt-Right of Way ...,Vacant Exempt Right of Way ...


In [20]:
prop[prop['Account_Num'].isin(['42722444', '01073451'])]

,RP,Appraisal_Year,Account_Num,Record_Type,Sequence_No,PIDN,Owner_Name,Owner_Address,Owner_CityState,Owner_Zip,Owner_Zip4,Owner_CRRT,Situs_Address,Property_Class,TAD_Map,MAPSCO,Exemption_Code,State_Use_Code,LegalDescription,Notice_Date,County,City,School,Num_Special_Dist,Spec1,Spec2,Spec3,Spec4,Spec5,Deed_Date,Deed_Book,Deed_Page,Land_Value,Improvement_Value,Total_Value,Garage_Capacity,Num_Bedrooms,Num_Bathrooms,Year_Built,Living_Area,Swimming_Pool_Ind,ARB_Indicator,Ag_Code,Land_Acres,Land_SqFt,Ag_Acres,Ag_Value,Central_Heat_Ind,Central_Air_Ind,Structure_Count,From_Accts,Appraisal_Date,Appraised_Value,GIS_Link,Instrument_No,Overlap_Flag
6024,C,2025,01073451,AAAA,000,15700----6---24A,HABITAT REHAB LLC,8305 THORNDYKE DR,"NORTH RICHLAND HILLS, TX",76182,,,2601 CARSON ST,F1,2072-408,065A,,F1,GOLDEN GARDENS ADDITION Block 6 Lot 24A,01-14-2026,220,027,902,2,000,224,225,000,000,08-03-2018,,,62976,573786,636762,0,0,0,2022,4500,,,,0.4130,17993,0.0000,000000000,N,N,0,,01-01-2025,636762,15700-6-24A,D218172463,
58076,C,2025,42722444,AAAA,000,A1622---1A02---60,FORT WORTH CITY OF,200 TEXAS ST,"FT WORTH, TX",76102,6311,,N CROWLEY RD,X,2048-340,118C,,X,"WALKER, HIRAM SURVEY Abstract 1622 Tract 1A02 ...",05-23-2025,220,026,912,3,223,224,225,000,000,11-28-2024,,,128471,0,128471,0,0,0,0,0,,,,7.4150,322998,0.0000,000000000,N,N,0,,01-01-2025,128471,A1622-1A02-60,D224213134,


In [23]:
prop[prop['Account_Num'].isin(['42722444', '01073451'])]

,RP,Appraisal_Year,Account_Num,Record_Type,Sequence_No,PIDN,Owner_Name,Owner_Address,Owner_CityState,Owner_Zip,Owner_Zip4,Owner_CRRT,Situs_Address,Property_Class,TAD_Map,MAPSCO,Exemption_Code,State_Use_Code,LegalDescription,Notice_Date,County,City,School,Num_Special_Dist,Spec1,Spec2,Spec3,Spec4,Spec5,Deed_Date,Deed_Book,Deed_Page,Land_Value,Improvement_Value,Total_Value,Garage_Capacity,Num_Bedrooms,Num_Bathrooms,Year_Built,Living_Area,Swimming_Pool_Ind,ARB_Indicator,Ag_Code,Land_Acres,Land_SqFt,Ag_Acres,Ag_Value,Central_Heat_Ind,Central_Air_Ind,Structure_Count,From_Accts,Appraisal_Date,Appraised_Value,GIS_Link,Instrument_No,Overlap_Flag
6024,C,2025,01073451,AAAA,000,15700----6---24A,HABITAT REHAB LLC,8305 THORNDYKE DR,"NORTH RICHLAND HILLS, TX",76182,,,2601 CARSON ST,F1,2072-408,065A,,F1,GOLDEN GARDENS ADDITION Block 6 Lot 24A,01-14-2026,220,027,902,2,000,224,225,000,000,08-03-2018,,,62976,573786,636762,0,0,0,2022,4500,,,,0.4130,17993,0.0000,000000000,N,N,0,,01-01-2025,636762,15700-6-24A,D218172463,
58076,C,2025,42722444,AAAA,000,A1622---1A02---60,FORT WORTH CITY OF,200 TEXAS ST,"FT WORTH, TX",76102,6311,,N CROWLEY RD,X,2048-340,118C,,X,"WALKER, HIRAM SURVEY Abstract 1622 Tract 1A02 ...",05-23-2025,220,026,912,3,223,224,225,000,000,11-28-2024,,,128471,0,128471,0,0,0,0,0,,,,7.4150,322998,0.0000,000000000,N,N,0,,01-01-2025,128471,A1622-1A02-60,D224213134,


In [22]:
cross_type

,RP,AppraisalYear,AccountNumber,AgentName,AgentNumber,Condition,DepreciationPercent,Latitude,Longitude,EYOC,NBHD,Quality,SiteName,SiteClassCd,SiteClassDescr,PartyId,LastSaleDate,SiteId,SitusCity,NAICSDescr,NAICSCd,PercentComplete,NoticeValue,SubdivisionName,NumberofStories,LandUseCode,Country,AddressType,SiteNumber
73201,R,2025,01073451,,,,,32.7983968588,-97.2562176249,,3H030A ...,,GOLDEN GARDENS ADDITION-6-24A ...,C1,Residential - Vacant Land ...,773951,2018-08-03,101073451,,,,0%,,GOLDEN GARDENS ADDITION,0,Residential 3H ...,UNITED STATES OF AMERICA,Standard,NaN
618567,R,2025,42722444,,,,,32.5988813079,-97.3458790668,,4B030H ...,,"JENNINGS, J SURVEY Abstract 875 Tract 2A",ResAg,Residential - Agricultural ...,1866,2024-11-28,80300758,,,,0%,128472,"WALKER, HIRAM SURVEY",0,Residential 4B1 ...,UNITED STATES OF AMERICA,Standard,NaN
664236,C,2025,01073451,,,Excellent ...,,32.7983968588,-97.2562176249,2022,WH-Midway ...,Good ...,I&A ROOFING SUPPLY CO,WHStorage,Warehouse-Storage ...,773951,2018-08-03,101073451,HALTOM CITY,,,100%,636762,GOLDEN GARDENS ADDITION,1,Warehouse-Non Business Park ...,UNITED STATES OF AMERICA,Standard,01073451
714289,C,2025,42722444,,,,,32.5988813079,-97.3458790668,,Right Of Way General ...,,"WALKER, HIRAM SURVEY Abstract 1622 Tract 1A02 ...",ExROW,Exempt-Right of Way ...,1866,2024-11-28,941740087,FORT WORTH,,,0%,128472,"WALKER, HIRAM SURVEY",0,Vacant Exempt Right of Way ...,UNITED STATES OF AMERICA,Standard,800108411


In [18]:
cross_type

,RP,AppraisalYear,AccountNumber,AgentName,AgentNumber,Condition,DepreciationPercent,Latitude,Longitude,EYOC,NBHD,Quality,SiteName,SiteClassCd,SiteClassDescr,PartyId,LastSaleDate,SiteId,SitusCity,NAICSDescr,NAICSCd,PercentComplete,NoticeValue,SubdivisionName,NumberofStories,LandUseCode,Country,AddressType,SiteNumber
73201,R,2025,01073451,,,,,32.7983968588,-97.2562176249,,3H030A ...,,GOLDEN GARDENS ADDITION-6-24A ...,C1,Residential - Vacant Land ...,773951,2018-08-03,101073451,,,,0%,,GOLDEN GARDENS ADDITION,0,Residential 3H ...,UNITED STATES OF AMERICA,Standard,NaN
618567,R,2025,42722444,,,,,32.5988813079,-97.3458790668,,4B030H ...,,"JENNINGS, J SURVEY Abstract 875 Tract 2A",ResAg,Residential - Agricultural ...,1866,2024-11-28,80300758,,,,0%,128472,"WALKER, HIRAM SURVEY",0,Residential 4B1 ...,UNITED STATES OF AMERICA,Standard,NaN
664236,C,2025,01073451,,,Excellent ...,,32.7983968588,-97.2562176249,2022,WH-Midway ...,Good ...,I&A ROOFING SUPPLY CO,WHStorage,Warehouse-Storage ...,773951,2018-08-03,101073451,HALTOM CITY,,,100%,636762,GOLDEN GARDENS ADDITION,1,Warehouse-Non Business Park ...,UNITED STATES OF AMERICA,Standard,01073451
714289,C,2025,42722444,,,,,32.5988813079,-97.3458790668,,Right Of Way General ...,,"WALKER, HIRAM SURVEY Abstract 1622 Tract 1A02 ...",ExROW,Exempt-Right of Way ...,1866,2024-11-28,941740087,FORT WORTH,,,0%,128472,"WALKER, HIRAM SURVEY",0,Vacant Exempt Right of Way ...,UNITED STATES OF AMERICA,Standard,800108411


## 4. Join

In [7]:
joined = prop.merge(
    supp,
    how="outer",
    left_on=PROP_KEY,
    right_on=SUPP_KEY,
    validate="one_to_one",
    indicator=True,
)

# Supplemental rows with no PropertyData match, kept aside for review
supp_only = joined[joined["_merge"] == "right_only"][supp.columns]
print(f"Supplemental rows with no PropertyData match: {len(supp_only):,}")
print(supp_only["RP"].value_counts())

joined = joined[joined["_merge"] != "right_only"]
joined.groupby(["RP", "_merge"], observed=True).size().unstack(fill_value=0)

Supplemental rows with no PropertyData match: 53
RP
R    53
Name: count, dtype: int64


_merge,left_only,both
RP,,
C,15,61111
M,201,1207563
P,0,68227
R,1005,658356


In [8]:
joined = joined.drop(columns=["AppraisalYear", "AccountNumber", "_merge"])
assert len(joined) == len(prop)
print(joined.shape)
joined.head()

(1996478, 82)


,RP,Appraisal_Year,Account_Num,Record_Type,Sequence_No,PIDN,Owner_Name,Owner_Address,Owner_CityState,Owner_Zip,Owner_Zip4,Owner_CRRT,Situs_Address,Property_Class,TAD_Map,MAPSCO,Exemption_Code,State_Use_Code,LegalDescription,Notice_Date,County,City,School,Num_Special_Dist,Spec1,Spec2,Spec3,Spec4,Spec5,Deed_Date,Deed_Book,Deed_Page,Land_Value,Improvement_Value,Total_Value,Garage_Capacity,Num_Bedrooms,Num_Bathrooms,Year_Built,Living_Area,Swimming_Pool_Ind,ARB_Indicator,Ag_Code,Land_Acres,Land_SqFt,Ag_Acres,Ag_Value,Central_Heat_Ind,Central_Air_Ind,Structure_Count,From_Accts,Appraisal_Date,Appraised_Value,GIS_Link,Instrument_No,Overlap_Flag,AgentName,AgentNumber,Condition,DepreciationPercent,Latitude,Longitude,EYOC,NBHD,Quality,SiteName,SiteClassCd,SiteClassDescr,PartyId,LastSaleDate,SiteId,SitusCity,NAICSDescr,NAICSCd,PercentComplete,NoticeValue,SubdivisionName,NumberofStories,LandUseCode,Country,AddressType,SiteNumber
0,C,2025,00000051,AAAA,000,14437----2----1,FORT WORTH CITY OF,200 TEXAS ST,"FT WORTH, TX",76102,6311,,117 E BELKNAP ST,C1C,2048-396,062Z,,C1,FORT WORTH ORIGINAL TOWN Block 2 Lot 1,,220,026,905,4,223,224,225,000,601,12-31-1900,0000000,0000000,450000,0,450000,0,0,0,0,0,,,,0.1147,5000,0.0000,000000000,N,N,0,,01-01-2025,450000,14437-2-1,0000000000,,,,,,32.7582971392,-97.3329220112,,Community Facility General ...,,115 E Belknap St Vacant Land,ExGovt,Exempt-Government ...,1866,1900-01-01,80873684,FORT WORTH,,,0%,,FORT WORTH ORIGINAL TOWN,0,Office-Mid to High Rise ...,UNITED STATES OF AMERICA,Standard,80873684
1,C,2025,00000078,AAAA,000,14437----2----2,FORT WORTH CITY OF,200 TEXAS ST,"FT WORTH, TX",76102,6311,,111 E BELKNAP ST,C1C,2048-396,062Z,,C1,FORT WORTH ORIGINAL TOWN Block 2 Lot 2,,220,026,905,4,223,224,225,000,601,12-31-1900,0000000,0000000,450000,0,450000,0,0,0,0,0,,,,0.1147,5000,0.0000,000000000,N,N,0,,01-01-2025,450000,14437-2-2,0000000000,,,,,,32.7582260964,-97.3330650169,,Community Facility General ...,,115 E Belknap St Vacant Land,ExGovt,Exempt-Government ...,1866,1900-01-01,80873684,FORT WORTH,,,0%,,FORT WORTH ORIGINAL TOWN,0,Office-Mid to High Rise ...,UNITED STATES OF AMERICA,Standard,80873684
2,C,2025,00000086,AAAA,000,14437----2----3A--20,FORT WORTH CITY OF,200 TEXAS ST,"FT WORTH, TX",76102,6311,,101 E BELKNAP ST,C1C,2048-396,062Z,,C1,FORT WORTH ORIGINAL TOWN Block 2 Lot 3A & 4,,220,026,905,4,223,224,225,000,601,12-31-1900,0000000,0000000,675000,0,675000,0,0,0,0,0,,,,0.1721,7500,0.0000,000000000,N,N,0,,01-01-2025,675000,14437-2-3A,0000000000,,,,,,32.7581077771,-97.3333141011,,Community Facility General ...,,115 E Belknap St Vacant Land,ExGovt,Exempt-Government ...,1866,1900-01-01,80873684,FORT WORTH,,,0%,,FORT WORTH ORIGINAL TOWN,0,Office-Mid to High Rise ...,UNITED STATES OF AMERICA,Standard,80873684
3,C,2025,00000094,AAAA,000,14437----2----3B,FORT WORTH CITY OF,200 TEXAS ST,"FT WORTH, TX",76102,6311,,115 E BELKNAP ST,C1C,2048-396,062Z,,C1,FORT WORTH ORIGINAL TOWN Block 2 Lot 3B,,220,026,905,4,223,224,225,000,601,12-31-1900,0000000,0000000,225000,0,225000,0,0,0,0,0,,,,0.0573,2500,0.0000,000000000,N,N,0,,01-01-2025,225000,14437-2-3B,0000000000,,,,,,32.7581770524,-97.3331625054,,Community Facility General ...,,115 E Belknap St Vacant Land,ExGovt,Exempt-Government ...,1866,1900-01-01,80873684,FORT WORTH,,,0%,,FORT WORTH ORIGINAL TOWN,0,Office-Mid to High Rise ...,UNITED STATES OF AMERICA,Standard,80873684
4,C,2025,00000108,AAAA,000,14437----3----1---20,FORT WORTH CITY OF,200 TEXAS ST,"FT WORTH, TX",76102,6311,,100 W BELKNAP ST,C1C,2048-396,062Z,,C1,FORT WORTH ORIGINAL TOWN Block 3 Lot 1 & 2,,220,026,905,4,223,224,225,000,601,12-31-1900,0000000,0000000,1080000,0,1080000,0,0,0,0,0,,,,0.2754,12000,0.0000,000000000,N,N,0,,01-01-2025,1080000,14437-3-1,0000000000,,,,,,32.7579260328,-97.3336783888,,OFC-Central Business District ...,,CITY OF FORT WORTH,ExGovt,Exempt-Government ...,1866,1900-01-01,80000088,FORT WORTH,,,0%,,FORT WORTH ORIGINAL TOWN,0,Community Facility ...,UNITED STATES OF AMERICA,Standard,80000088


In [13]:
joined[joined['Latitude'].isna()]

,RP,Appraisal_Year,Account_Num,Record_Type,Sequence_No,PIDN,Owner_Name,Owner_Address,Owner_CityState,Owner_Zip,Owner_Zip4,Owner_CRRT,Situs_Address,Property_Class,TAD_Map,MAPSCO,Exemption_Code,State_Use_Code,LegalDescription,Notice_Date,County,City,School,Num_Special_Dist,Spec1,Spec2,Spec3,Spec4,Spec5,Deed_Date,Deed_Book,Deed_Page,Land_Value,Improvement_Value,Total_Value,Garage_Capacity,Num_Bedrooms,Num_Bathrooms,Year_Built,Living_Area,Swimming_Pool_Ind,ARB_Indicator,Ag_Code,Land_Acres,Land_SqFt,Ag_Acres,Ag_Value,Central_Heat_Ind,Central_Air_Ind,Structure_Count,From_Accts,Appraisal_Date,Appraised_Value,GIS_Link,Instrument_No,Overlap_Flag,AgentName,AgentNumber,Condition,DepreciationPercent,Latitude,Longitude,EYOC,NBHD,Quality,SiteName,SiteClassCd,SiteClassDescr,PartyId,LastSaleDate,SiteId,SitusCity,NAICSDescr,NAICSCd,PercentComplete,NoticeValue,SubdivisionName,NumberofStories,LandUseCode,Country,AddressType,SiteNumber
1997,C,2025,00319503,AAAA,000,-3840----2----3,Current Owner,Current Address,N/A,00000,0000,,Current Address,F1,N/A,N/A,,F1,N/A,05-01-2025,N/A,N/A,N/A,3,223,224,225,000,000,N/A,N/A,N/A,160000,0,160000,0,0,0,1989,0,,,,0.1147,5000,0.0000,000000000,N,N,0,,01-01-2025,160000,3840-2-3,N/A,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1998,C,2025,00319619,AAAA,000,-3840----2---13---20,Current Owner,Current Address,N/A,00000,0000,,Current Address,F1,N/A,N/A,,F1,N/A,05-01-2025,N/A,N/A,N/A,3,223,224,225,000,000,N/A,N/A,N/A,352000,169446,521446,0,0,0,1989,4614,,,,0.2525,11000,0.0000,000000000,N,N,0,,01-01-2025,521446,3840-2-13,N/A,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2629,C,2025,00475084,AAAA,000,-6980---37----1R1,Current Owner,Current Address,N/A,00000,0000,,Current Address,F1,N/A,N/A,,F1,N/A,05-01-2025,N/A,N/A,N/A,4,223,224,225,000,639,N/A,N/A,N/A,489500,885177,1374677,0,0,0,1922,7616,,,,0.4494,19580,0.0000,000000000,N,N,0,,01-01-2025,1374677,6980-37-1R1,N/A,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8049,C,2025,01483595,AAAA,000,22320----2----1B,Current Owner,Current Address,N/A,00000,0000,,Current Address,F1,N/A,N/A,,F1,N/A,04-15-2025,N/A,N/A,N/A,2,000,224,225,000,000,N/A,N/A,N/A,51222,27945,79167,0,0,0,0,0,,,,0.1960,8537,0.0000,000000000,N,N,0,,01-01-2025,79167,22320-2-1B,N/A,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8051,C,2025,01483633,AAAA,000,22320----2----3A--20,Current Owner,Current Address,N/A,00000,0000,,Current Address,F1,N/A,N/A,,F1,N/A,05-01-2025,N/A,N/A,N/A,2,000,224,225,000,000,N/A,N/A,N/A,52800,1222200,1275000,0,0,0,1930,7318,,,,0.2020,8800,0.0000,000000000,N,N,0,,01-01-2025,1272000,22320-2-3A,N/A,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1996526,R,2025,43315537,AAAA,000,25270----6---13---51,"MIRANDA, LOUIE DUNGCA",300 CANNON DR,"HURST, TX",76054,,,300 CANNON DR,A1,,,,A,MAYFAIR NORTH ADDITION Block 6 Lot 13 50% UND...,,220,028,916,2,000,224,225,000,000,01-02-2023,,,37500,116988,154488,2,0,0,1977,1656,,,,0.2666,11616,0.0000,000000000,Y,Y,0,,01-01-2025,154488,25270-6-13,02D2192115,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1996527,R,2025,43315766,AAAA,000,48030---13--10B---51,"UNDERWOOD, JOHN",7821 SOMMERVILLE PL,"FORT WORTH, TX",76135,,,7821 SOMMERVILLE PLACE RD,A1,,,,A,"YOUNG, ELLA ADDITION Block 13 Lot 10B 50% UND...",,220,015,915,3,223,224,225,000,000,01-02-2024,,,19838,88060,107898,3,0,0,1959,1112,,,,0.2644,11

## 5. Export

In [ ]:
joined.to_csv(OUTPUT_FILE, index=False)
print(OUTPUT_FILE.resolve(), f"{OUTPUT_FILE.stat().st_size / 1e6:,.1f} MB")